# TP1 — Multiple Linear Regression
Working in pairs is acceptable; however, I strongly recommend submitting the assignment individually. The deadline for this first assignment is February 16, 2026. Please send your response to razan.mhanna@inria.fr
 only.


**Dataset:** Diabetes (regression)

## Learning objectives
- Construct a design matrix and add an intercept
- Fit a multiple linear regression model (OLS)
- Interpret coefficients, p-values, and $R^2$
- Make predictions with confidence intervals
- Perform basic diagnostic checks


## Question 1 — Load the dataset
1. Load the **Diabetes** regression dataset.
2. Convert it to a pandas DataFrame.
3. Identify:
   - the number of observations $n$
   - the number of predictors $p$

👉 *Write the code below.*

In [1]:
!pip -q install statsmodels scikit-learn pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.datasets import load_diabetes
pd.set_option('display.precision', 4)
print('Ready!')


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Ready!


In [2]:
diab = load_diabetes(as_frame=True)
df = diab.frame.copy()

In [3]:
n, p = df.shape[0], df.shape[1] - 1
n, p


(442, 10)

## Question 2 — Define $X$ and $y$
1. Define the response variable $y$.
2. Define the predictor matrix $X$.
3. Add an intercept column to $X$.

👉 *Explain why adding a constant is necessary when using statsmodels.*

In [4]:
y = df["target"]
X = sm.add_constant(df.drop(columns="target"))
X.head()


,const,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,1.0,0.0381,0.0507,0.0617,0.0219,-0.0442,-0.0348,-0.0434,-0.0026,0.0199,-0.0176
1,1.0,-0.0019,-0.0446,-0.0515,-0.0263,-0.0084,-0.0192,0.0744,-0.0395,-0.0683,-0.0922
2,1.0,0.0853,0.0507,0.0445,-0.0057,-0.0456,-0.0342,-0.0324,-0.0026,0.0029,-0.0259
3,1.0,-0.0891,-0.0446,-0.0116,-0.0367,0.0122,0.0250,-0.0360,0.0343,0.0227,-0.0094
4,1.0,0.0054,-0.0446,-0.0364,0.0219,0.0039,0.0156,0.0081,-0.0026,-0.0320,-0.0466


Statsmodels does not add an intercept automatically in `OLS`. Adding a constant creates the column for $\beta_0$, so the model estimates $\hat{y} = \beta_0 + \beta_1x_1 + \cdots + \beta_px_p$ instead of forcing the regression hyperplane through the origin.

## Question 3 — Fit the multiple linear regression model
1. Fit an **OLS** model using statsmodels.
2. Display the full regression summary.

👉 *Write down the estimated regression equation.*

In [5]:
model = sm.OLS(y, X).fit()
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 target   R-squared:                       0.518
Model:                            OLS   Adj. R-squared:                  0.507
Method:                 Least Squares   F-statistic:                     46.27
Date:                Fri, 12 Jun 2026   Prob (F-statistic):           3.83e-62
Time:                        11:39:45   Log-Likelihood:                -2386.0
No. Observations:                 442   AIC:                             4794.
Df Residuals:                     431   BIC:                             4839.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        152.1335      2.576     59.061      0.000     147.071     157.196
age          -10.0099     59.749     -0.168      0.867    -127.446     107.426
sex         -239.8156     61.222     -3.917      0.000    -360.147    -119.484
bmi          519.8459     66.533      7.813      0.000     389.076     650.616
bp           324.3846     65.422      4.958      0.000     195.799     452.970
s1          -792.1756    416.680     -1.901      0.058   -1611.153      26.802
s2           476.7390    339.030      1.406      0.160    -189.620    1143.098
s3           101.0433    212.531      0.475      0.635    -316.684     518.770
s4           177.0632    161.476      1.097      0.273    -140.315     494.441
s5           751.2737    171.900      4.370      0.000     413.407    1089.140
s6            67.6267     65.984      1.025      0.306     -62.064     197.318
==============================================================================
Omnibus:                        1.506   Durbin-Watson:                   2.029
Prob(Omnibus):                  0.471   Jarque-Bera (JB):                1.404
Skew:                           0.017   Prob(JB):                        0.496
Kurtosis:                       2.726   Cond. No.                         227.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

The estimated regression equation is $\widehat{target} = 152.13 - 10.01\,age - 239.82\,sex + 519.85\,bmi + 324.38\,bp - 792.18\,s1 + 476.74\,s2 + 101.04\,s3 + 177.06\,s4 + 751.27\,s5 + 67.63\,s6$.

## Question 4 — Interpretation of coefficients
Using the regression output:
1. Which predictors are statistically significant at the 5% level?
2. Interpret **one significant coefficient** in words.
3. Is the intercept meaningful in this context? Explain.

At the 5% level, the significant predictors are `sex`, `bmi`, `bp`, and `s5`, because their p-values are below $0.05$. The coefficient of `bmi` is about $519.85$, so for a one-unit increase in the standardized BMI variable, the predicted diabetes progression score increases by about $519.85$ units when all other predictors are held fixed. The intercept has limited clinical meaning: since the predictors are standardized, it estimates the expected target for an observation at the mean of every predictor, not a patient whose raw measurements are zero.

## Question 5 — Goodness of fit
1. What is the value of $R^2$?
2. What does $R^2$ represent in this model?
3. Compare $R^2$ and adjusted $R^2$. Why are they different?

The model has $R^2 = 0.5177$, so it explains about $51.8\%$ of the variability in the diabetes target using these ten predictors. The adjusted value is $0.5066$, which is smaller because it penalizes the inclusion of predictors and accounts for the sample size and number of predictors. They differ because ordinary $R^2$ cannot decrease when predictors are added, while adjusted $R^2$ only increases when the added predictors improve the fit enough to justify their cost in degrees of freedom.

## Question 6 — Multicollinearity
1. Compute the correlation matrix of the predictors.
2. Identify any strongly correlated variables.

👉 *Explain how multicollinearity affects coefficient estimates.*

In [6]:
corr = df.drop(columns="target").corr()
corr


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
age,1.0000,0.1737,0.1851,0.3354,0.2601,0.2192,-0.0752,0.2038,0.2708,0.3017
sex,0.1737,1.0000,0.0882,0.2410,0.0353,0.1426,-0.3791,0.3321,0.1499,0.2081
bmi,0.1851,0.0882,1.0000,0.3954,0.2498,0.2612,-0.3668,0.4138,0.4462,0.3887
bp,0.3354,0.2410,0.3954,1.0000,0.2425,0.1855,-0.1788,0.2577,0.3935,0.3904
s1,0.2601,0.0353,0.2498,0.2425,1.0000,0.8967,0.0515,0.5422,0.5155,0.3257
s2,0.2192,0.1426,0.2612,0.1855,0.8967,1.0000,-0.1965,0.6598,0.3184,0.2906
s3,-0.0752,-0.3791,-0.3668,-0.1788,0.0515,-0.1965,1.0000,-0.7385,-0.3986,-0.2737
s4,0.2038,0.3321,0.4138,0.2577,0.5422,0.6598,-0.7385,1.0000,0.6179,0.4172
s5,0.2708,0.1499,0.4462,0.3935,0.5155,0.3184,-0.3986,0.6179,1.0000,0.4647
s6,0.3017,0.2081,0.3887,0.3904,0.3257,0.2906,-0.2737,0.4172,0.4647,1.0000


Using an absolute correlation around $0.7$ as a practical threshold, the strongest relationships are between `s1` and `s2`, with correlation about $0.897$, and between `s3` and `s4`, with correlation about $-0.738$. Multicollinearity does not usually make the fitted predictions invalid, but it makes individual coefficient estimates less stable and increases their standard errors, so p-values and signs for correlated predictors can become harder to interpret.

## Question 7 (Bonus) — Global significance of the model
1. State the null hypothesis tested by the **F-statistic**.
2. Based on the p-value, is the model globally significant?
3. What does this tell you about the predictors as a group?

In [7]:
model.fvalue, model.f_pvalue


(np.float64(46.27243958524319), np.float64(3.8286490381855213e-62))

The F-test has null hypothesis $H_0: \beta_{age} = \beta_{sex} = \beta_{bmi} = \beta_{bp} = \beta_{s1} = \beta_{s2} = \beta_{s3} = \beta_{s4} = \beta_{s5} = \beta_{s6} = 0$, meaning none of the predictors has a linear effect once considered together. The p-value is about $3.83 \times 10^{-62}$, which is far below $0.05$, so the model is globally significant. This indicates that the predictors as a group explain a statistically significant part of the variation in the diabetes target.